<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%205/Aula_5_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 5.3 — Governança, escala e boas práticas

**Curso:** Criando Agentes de IA com Agno

**Aula:** 5 — Gerenciando agentes com AgentOS

**Instrutor:** Fabio Contrera

---

## Onde estamos

Esta é a **última aula do curso**.


1. **Versionamento de prompts**
2. **Escala**
3. **LGPD na prática**
4. **Boas práticas operacionais**
5. **Fechamento do curso**


## Setup

Sem dependências novas — `agno[os]` já traz o `PIIDetectionGuardrail` que vamos usar.


In [ ]:
!pip install -U "agno[os]" openai

In [6]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

---

## Passo 1 — Versionamento de prompts




In [7]:
import yaml
from pathlib import Path

# Criar um arquivo de prompts versionado
prompts_yaml = '''
treinador:
  v1:
    description: "Versão inicial — funcional mas genérica"
    instructions:
      - "Você é o Treinador do ScoutAI FC."
      - "Responda sobre futebol em português do Brasil."

  v2:
    description: "Adicionado tom profissional e tratamento de incerteza"
    instructions:
      - "Você é o Treinador do ScoutAI FC."
      - "Responda sobre futebol em português do Brasil, com tom profissional e analítico."
      - "Quando não tiver certeza de um dado, diga claramente."

  v3_atual:
    description: "Adicionado uso de personalização (memória) — versão atual em produção"
    instructions:
      - "Você é o Treinador do ScoutAI FC."
      - "Responda sobre futebol em português do Brasil, com tom profissional e analítico."
      - "Quando não tiver certeza de um dado, diga claramente."
      - "Quando souber preferências do usuário (time, estilo de jogo), considere ao recomendar."
'''

# Salvar e carregar
Path("/tmp/prompts.yaml").write_text(prompts_yaml)
prompts = yaml.safe_load(Path("/tmp/prompts.yaml").read_text())

# Listar versões disponíveis
print("Versões do prompt 'treinador':\n")
for versao, conteudo in prompts["treinador"].items():
    print(f"  {versao}: {conteudo['description']}")

Versões do prompt 'treinador':

  v1: Versão inicial — funcional mas genérica
  v2: Adicionado tom profissional e tratamento de incerteza
  v3_atual: Adicionado uso de personalização (memória) — versão atual em produção


### Carregando uma versão específica no agente

Com o YAML carregado, o agente recebe instructions de uma versão específica. Pra trocar de versão em produção, você muda **uma string** — não 30 linhas de código:


In [8]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb

VERSAO_ATUAL = "v3_atual"   # ← apenas isso muda quando você troca de versão

db = SqliteDb(db_file="/tmp/scoutai_5_3.db")

modelo_treinador = OpenAIChat(id="gpt-5.4")

treinador = Agent(
    name="Treinador",
    model= modelo_treinador,
    db=db,
    instructions=prompts["treinador"][VERSAO_ATUAL]["instructions"],
    add_history_to_context=True,
    enable_user_memories=True,
    markdown=True,
)

print(f"Treinador rodando com prompt versão {VERSAO_ATUAL}")
print(f"Descrição: {prompts['treinador'][VERSAO_ATUAL]['description']}")

Treinador rodando com prompt versão v3_atual
Descrição: Adicionado uso de personalização (memória) — versão atual em produção


### Por que isso vira essencial em produção

- **Rollback instantâneo**
- **A/B testing**
- **Auditoria**




---

## Passo 2 — LGPD na prática: protegendo PII com guardrails nativos


O risco real:

- Usuário escreve *"Sou o Fabio Silva, CPF 123.456.789-00, e quero..."*
- Agente extrai isso como **memória de longo prazo** e guarda no banco
- Banco vaza, vira incidente sob LGPD



### Modo 1 — bloquear input com PII

A forma mais restritiva: se o input contém PII, o agente **rejeita antes de chamar o LLM**. Útil quando a política é "não processamos dados sensíveis nunca".


In [9]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.guardrails import PIIDetectionGuardrail

# Agente com guardrail no modo bloqueio (default)
treinador_bloqueio = Agent(
    name="Treinador",
    model=OpenAIChat(id="gpt-5.4-mini"),
    pre_hooks=[PIIDetectionGuardrail()],
    instructions=["Você é o Treinador do ScoutAI FC. Responda em português."],
    markdown=True,
)

# Tenta enviar input com email — deve ser bloqueado
try:
    treinador_bloqueio.print_response(
        "Oi, sou o Fabio. Meu email é fabio.contrera@scoutai.fc — pode me ajudar?",
        stream=False,
    )
except Exception as e:
    print(f"❌ Bloqueado pelo guardrail: {type(e).__name__}")
    print(f"   Mensagem: {e}")

ERROR    Validation failed: Potential PII detected in input | Check trigger: CheckTrigger.PII_DETECTED

### Modo 2 — mascarar PII em vez de bloquear

Mais flexível: o guardrail **substitui o PII por máscara** antes do LLM ver. Conversa flui (agente entende contexto), mas dado pessoal não vaza.


In [10]:
# Mesmo agente, mas configurado para mascarar
treinador_mascara = Agent(
    name="Treinador",
    model=OpenAIChat(id="gpt-5.4-mini"),
    pre_hooks=[PIIDetectionGuardrail(mask_pii=True)],
    instructions=["Você é o Treinador do ScoutAI FC. Responda em português."],
    markdown=True,
)

# Mesmo input — agora processa, mas o LLM vê o email mascarado
treinador_mascara.print_response(
    "Oi, sou o Fabio. Meu email é fabio.contrera@scoutai.fc — pode me ajudar?",
    stream=True,
)

Output()

O LLM **respondeu sem ter visto o email original**. Recebeu algo como `"meu email é ********************"`. A conversa continuou; o dado sensível não saiu do escopo do servidor.


### Modo 3 — padrões customizados para LGPD brasileira




In [11]:
import re

# Guardrail estendido com padrões brasileiros
guardrail_brasil = PIIDetectionGuardrail(
    mask_pii=True,
    custom_patterns={
        "cpf": re.compile(r"\d{3}\.\d{3}\.\d{3}-\d{2}"),
        "rg": re.compile(r"\d{2}\.\d{3}\.\d{3}-\d{1}"),
    },
)

treinador_lgpd = Agent(
    name="Treinador",
    model=OpenAIChat(id="gpt-5.4-mini"),
    pre_hooks=[guardrail_brasil],
    instructions=["Você é o Treinador do ScoutAI FC. Responda em português."],
    markdown=True,
)

# Input com CPF — deve ser mascarado
treinador_lgpd.print_response(
    "Oi, sou o Fabio Silva, CPF 123.456.789-00. Quero saber sobre a Seleção.",
    stream=True,
)

Output()

---

## Passo 3 — Boas práticas operacionais



### **1. Evals contínuos**


### **2. Monitoramento contínuo**

- Latência por turno
- Custo por usuário
- Taxa de uso de tools
- Erros silenciosos

### **3. Guardrails**

- Pre-hooks
- Post-hooks
- Validação estrutural
- Limite máximo de tokens
- Lista de tópicos vetados
- Confirmação humana


### **4. Versionamento de tudo**



---

## Passo 4 — Recap do curso


| Aula | O que este curso entregou |
|---|---|
| **1** | Conceito de agente, primeiro Treinador no Agno |
| **2** |  Tools (Tavily + Wikipedia), política de fontes |
| **3.1** | Time mínimo (Treinador + Olheiro) |
| **3.2** |  Analista + tools customizadas + memória de sessão |
| **3.3** |  RAG no Olheiro + refinamento de delegação |
| **4.1** |  Workflow básico (`Workflow`, `Step`) |
| **4.2** |  Pydantic + Auxiliar Técnico + `Parallel` |
| **4.3** |  `Loop` + `Condition` + reasoning + roteador time/workflow |
| **5.1** |  AgentOS + persistência (`SqliteDb`) |
| **5.2** |  `enable_user_memories` + observabilidade Python |
| **5.3** |  Versionamento + escala + LGPD + boas práticas |

